In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install opencv-python scikit-image tensorflow pandas

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Concatenate
from sklearn.model_selection import train_test_split

2026-03-05 14:07:43.278740: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772719663.607533      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772719663.781198      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772719664.622100      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772719664.622155      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772719664.622158      55 computation_placer.cc:177] computation placer alr

In [4]:
csv_path = "/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv"

df = pd.read_csv(csv_path)

print(df.head())

      image_name  thickness  label  True_age       age  gender  group
0  2491006_R.png        0.8      0        63  0.684932       0      1
1  2491006_L.png        0.8      0        63  0.684932       0      1
2  3730004_R.png        1.2      1        61  0.657534       1      1
3  3730004_L.png        1.2      1        61  0.657534       1      1
4  3730006_R.png        1.2      1        64  0.698630       1      1


In [5]:
image_dir = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"

df["path"] = image_dir + df["image_name"]

In [6]:
def crop_image(img):
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    coords = np.column_stack(np.where(gray > 10))
    
    x,y,w,h = cv2.boundingRect(coords)
    
    crop = img[y:y+h, x:x+w]
    
    return crop

In [7]:
IMG_SIZE = 224

def preprocess(path):
    
    img = cv2.imread(path)
    
    img = crop_image(img)
    
    img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    
    img = img/255.0
    
    return img

In [8]:
def build_unet(input_shape=(224,224,3)):

    inputs = Input(input_shape)

    c1 = layers.Conv2D(64,3,activation='relu',padding='same')(inputs)
    c1 = layers.Conv2D(64,3,activation='relu',padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(128,3,activation='relu',padding='same')(p1)
    c2 = layers.Conv2D(128,3,activation='relu',padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    b = layers.Conv2D(256,3,activation='relu',padding='same')(p2)

    u1 = layers.UpSampling2D()(b)
    m1 = layers.concatenate([u1,c2])
    c3 = layers.Conv2D(128,3,activation='relu',padding='same')(m1)

    u2 = layers.UpSampling2D()(c3)
    m2 = layers.concatenate([u2,c1])
    c4 = layers.Conv2D(64,3,activation='relu',padding='same')(m2)

    outputs = layers.Conv2D(1,1,activation='sigmoid')(c4)

    return Model(inputs,outputs)

In [9]:
unet = build_unet()

unet.load_weights("/kaggle/input/vesselmodel/unet_vessel.h5")

I0000 00:00:1772719697.415846      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/kaggle/input/vesselmodel/unet_vessel.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
def get_vessel_mask(img):

    img = np.expand_dims(img,0)

    pred = unet.predict(img)[0]

    mask = (pred > 0.5).astype("float32")

    return mask

In [ ]:
def vessel_density(mask):

    density = np.sum(mask)/mask.size

    return density

In [ ]:
images = []
clinical = []
targets = []

for i,row in df.iterrows():

    img = preprocess(row["path"])

    mask = get_vessel_mask(img)

    density = vessel_density(mask)

    images.append(img)

    clinical.append([row["age"],row["gender"],density])

    targets.append(row["thickness"])

images = np.array(images)
clinical = np.array(clinical)
targets = np.array(targets)

In [ ]:
X_img_train, X_img_test, X_clin_train, X_clin_test, y_train, y_test = train_test_split(
    
    images, clinical, targets, test_size=0.2, random_state=42
)

In [ ]:
image_input = Input(shape=(224,224,3))

base_model = ResNet50(weights='imagenet',include_top=False,input_tensor=image_input)

x = GlobalAveragePooling2D()(base_model.output)

clinical_input = Input(shape=(3,))

y = Dense(32,activation="relu")(clinical_input)

fusion = Concatenate()([x,y])

In [ ]:
cimt_output = Dense(1,name="cimt")(fusion)

cvd_output = Dense(1,activation="sigmoid",name="cvd")(fusion)

model = Model(inputs=[image_input,clinical_input],
              outputs=[cimt_output,cvd_output])

In [ ]:
model.compile(

optimizer="adam",

loss={
"cimt":"mse",
"cvd":"binary_crossentropy"
},

metrics=["accuracy"]
)

In [ ]:
model.fit(

[X_img_train,X_clin_train],

[y_train,y_train>0.9],

epochs=30,

batch_size=16,

validation_split=0.1
)

In [ ]:
model.evaluate(

[X_img_test,X_clin_test],

[y_test,y_test>0.9]
)